In [1]:
pip install requests pandas numpy


In [2]:
import requests
import pandas as pd
import numpy as np
import time
from pathlib import Path

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
# =========================
# RESEARCH PARAMETERS
# =========================

TICKER = input("Enter the Yahoo Finance ticker, for example AAPL: ").strip().upper()
START_DATE = input("Enter start date in YYYY-MM-DD format: ").strip()
END_DATE = input("Enter end date in YYYY-MM-DD format: ").strip()

# Example:
# ticker = "AAPL"
# start_date = "2026-07-16"
# end_date = "2026-08-11"

# SEC requires a descriptive User-Agent
USER_AGENT = "PhD Research a.antonov1458@student.nu.edu"

OUTPUT_FOLDER = "sec_data"

Enter the Yahoo Finance ticker, for example AAPL: MSFT
Enter start date in YYYY-MM-DD format: 2026-07-16
Enter end date in YYYY-MM-DD format: 2026-08-11


In [9]:
#Create an SEC request function

HEADERS = {
    "User-Agent": USER_AGENT,
    "Accept-Encoding": "gzip, deflate",
    "Host": "data.sec.gov"
}


def sec_get_json(url, sleep_seconds=0.15):
    """
    Send a request to an SEC JSON endpoint.
    """

    response = requests.get(
        url,
        headers=HEADERS,
        timeout=30
    )

    response.raise_for_status()

    time.sleep(sleep_seconds)

    return response.json()

In [10]:
def get_company_cik(ticker):
    """
    Convert a stock ticker to SEC CIK.
    """

    url = "https://www.sec.gov/files/company_tickers.json"

    headers = {
        "User-Agent": USER_AGENT
    }

    response = requests.get(
        url,
        headers=headers,
        timeout=30
    )

    response.raise_for_status()

    ticker_data = response.json()

    ticker = ticker.upper()

    for _, company in ticker_data.items():

        if company["ticker"].upper() == ticker:

            cik = str(company["cik_str"]).zfill(10)

            return {
                "ticker": company["ticker"],
                "company": company["title"],
                "cik": cik
            }

    raise ValueError(f"Ticker {ticker} not found.")

In [11]:
#Testing
company = get_company_cik(TICKER)

print(company)

{'ticker': 'MSFT', 'company': 'MICROSOFT CORP', 'cik': '0000789019'}


In [12]:
#Download SEC filing history
def get_recent_filings(cik):
    """
    Retrieve recent SEC filing history for a company.
    """

    url = f"https://data.sec.gov/submissions/CIK{cik}.json"

    data = sec_get_json(url)

    filings = pd.DataFrame(
        data["filings"]["recent"]
    )

    return filings, data

In [13]:
#Test
cik = company["cik"]

filings, company_metadata = get_recent_filings(cik)

print(filings.head())

        accessionNumber  filingDate  reportDate        acceptanceDateTime act  \
0  0000789019-26-000143  2026-08-06  2026-08-05  2026-08-06T22:01:48.000Z       
1  0000789019-26-000141  2026-08-05  2026-08-04  2026-08-05T22:08:55.000Z       
2  0001959173-26-005674  2026-08-05              2026-08-05T19:35:10.000Z  33   
3  0001959173-26-005608  2026-08-04              2026-08-04T19:19:00.000Z  33   
4  0001193125-26-323660  2026-07-29  2026-06-30  2026-07-29T20:08:01.000Z  34   

   form fileNumber filmNumber items core_type      size  isXBRL  isInlineXBRL  \
0     4                                     4      5005       0             0   
1     4                                     4      5205       0             0   
2   144  001-37845  261243071             144      5350       0             0   
3   144  001-37845  261237954             144      5353       0             0   
4  10-K  001-37845  261217433            XBRL  36328553       1             1   

   isXBRLNumeric          

In [14]:
def filter_filings_by_date(
    filings,
    start_date,
    end_date
):

    df = filings.copy()

    df["filingDate"] = pd.to_datetime(
        df["filingDate"]
    )

    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)

    df = df[
        (df["filingDate"] >= start_date) &
        (df["filingDate"] <= end_date)
    ]

    return df.sort_values("filingDate").reset_index(drop=True)

In [15]:
#test
sec_filings = filter_filings_by_date(
    filings,
    START_DATE,
    END_DATE
)

print(sec_filings.shape)

sec_filings.head()

(7, 16)


,accessionNumber,filingDate,reportDate,acceptanceDateTime,act,form,fileNumber,filmNumber,items,core_type,size,isXBRL,isInlineXBRL,isXBRLNumeric,primaryDocument,primaryDocDescription
0,0001214659-26-008806,2026-07-22,,2026-07-22T14:59:22.000Z,34,PX14A6G,001-37845,261192530,,PX14A6G,46585,0,0,0.0,w721267px14a6g.htm,
1,0001193125-26-323660,2026-07-29,2026-06-30,2026-07-29T20:08:01.000Z,34,10-K,001-37845,261217433,,XBRL,36328553,1,1,1.0,msft-20260630.htm,10-K
2,0001193125-26-323632,2026-07-29,2026-07-29,2026-07-29T20:04:53.000Z,34,8-K,001-37845,261217337,"2.02,9.01",XBRL,1150858,1,1,0.0,msft-20260729.htm,8-K
3,0001959173-26-005608,2026-08-04,,2026-08-04T19:19:00.000Z,33,144,001-37845,261237954,,144,5353,0,0,0.0,xsl144X01/primary_doc.xml,
4,0000789019-26-000141,2026-08-05,2026-08-04,2026-08-05T22:08:55.000Z,,4,,,,4,5205,0,0,0.0,xslF345X06/form4.xml,STATEMENT OF CHANGES IN BENEFICIAL OWNERSHIP O...


In [16]:
#Select repotation-relevant forms

REPUTATION_FORMS = [
    "8-K",
    "8-K/A",

    "10-K",
    "10-K/A",

    "10-Q",
    "10-Q/A",

    "DEF 14A",

    "S-1",
    "S-1/A",

    "SC 13D",
    "SC 13D/A",

    "SC 13G",
    "SC 13G/A"
]

In [17]:
reputation_filings = sec_filings[
    sec_filings["form"].isin(REPUTATION_FORMS)
].copy()

print(
    reputation_filings[
        [
            "filingDate",
            "form",
            "items",
            "primaryDocDescription"
        ]
    ]
)

  filingDate  form      items primaryDocDescription
1 2026-07-29  10-K                             10-K
2 2026-07-29   8-K  2.02,9.01                   8-K


I would especially retain all 8-K filings.

An 8-K frequently represents an identifiable company event, which makes it particularly valuable for our event-based research design.

In [18]:
#Sec event table

def create_sec_event_dataset(
    filings,
    ticker,
    company_name,
    cik
):

    df = filings.copy()

    df["ticker"] = ticker
    df["company"] = company_name
    df["cik"] = cik

    df["source"] = "SEC_EDGAR"
    df["signal_layer"] = "Governance"

    df["event_date"] = pd.to_datetime(
        df["filingDate"]
    )

    df["event_type"] = (
        "SEC_" + df["form"].astype(str)
    )

    # Remove dashes for SEC archive URL
    df["accession_clean"] = (
        df["accessionNumber"]
        .str.replace("-", "", regex=False)
    )

    cik_no_zeros = str(int(cik))

    df["filing_url"] = (
        "https://www.sec.gov/Archives/edgar/data/"
        + cik_no_zeros
        + "/"
        + df["accession_clean"]
        + "/"
        + df["primaryDocument"]
    )

    columns = [
        "ticker",
        "company",
        "cik",

        "event_date",
        "source",
        "signal_layer",

        "event_type",
        "form",

        "items",

        "accessionNumber",

        "reportDate",
        "acceptanceDateTime",

        "primaryDocument",
        "primaryDocDescription",

        "isXBRL",
        "isInlineXBRL",

        "filing_url"
    ]

    return df[
        [c for c in columns if c in df.columns]
    ]

In [19]:
#test
sec_events = create_sec_event_dataset(
    reputation_filings,
    TICKER,
    company["company"],
    company["cik"]
)

sec_events.head(20)

,ticker,company,cik,event_date,source,signal_layer,event_type,form,items,accessionNumber,reportDate,acceptanceDateTime,primaryDocument,primaryDocDescription,isXBRL,isInlineXBRL,filing_url
1,MSFT,MICROSOFT CORP,0000789019,2026-07-29,SEC_EDGAR,Governance,SEC_10-K,10-K,,0001193125-26-323660,2026-06-30,2026-07-29T20:08:01.000Z,msft-20260630.htm,10-K,1,1,https://www.sec.gov/Archives/edgar/data/789019...
2,MSFT,MICROSOFT CORP,0000789019,2026-07-29,SEC_EDGAR,Governance,SEC_8-K,8-K,"2.02,9.01",0001193125-26-323632,2026-07-29,2026-07-29T20:04:53.000Z,msft-20260729.htm,8-K,1,1,https://www.sec.gov/Archives/edgar/data/789019...


In [20]:
#Export dataset
OUTPUT_FOLDER = Path("/content/drive/MyDrive/ColabNotebooks/DataForReputation")
#OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)


Path(OUTPUT_FOLDER).mkdir(
    parents=True,
    exist_ok=True
)

output_file = (
    f"{OUTPUT_FOLDER}/"
    f"{TICKER}_SEC_"
    f"{START_DATE}_"
    f"{END_DATE}.csv"
)

sec_events.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print(
    f"Saved {len(sec_events)} SEC events to:"
)

print(output_file)
print(f"CSV saved to: {output_file}")

Saved 2 SEC events to:
/content/drive/MyDrive/ColabNotebooks/DataForReputation/MSFT_SEC_2026-07-16_2026-08-11.csv
CSV saved to: /content/drive/MyDrive/ColabNotebooks/DataForReputation/MSFT_SEC_2026-07-16_2026-08-11.csv
